# F30K_T1S1_Seq paired non-inferiority

Kiểm định bootstrap ghép cặp 20.000 lần theo `image_id` để giải quyết cảnh báo độ ổn định caption sau Full Validation.

## Input bắt buộc

1. output `MOB_F30K_T1S1_Seq_INT8_Full_Validation`;
2. package/dataset `pycocoevalcap-1.2`.

Runtime: **CPU / Accelerator None / Internet Off**. Test không được đọc.


In [1]:
import hashlib, importlib.util, json, os, subprocess, sys
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd

MODEL_ID, TEACHER, STUDENT, METHOD, SEED = ('F30K_T1S1_Seq', 'ExpansionNet v2', 'MobileNetV3-Large', 'Sequence KD', 123)
SOURCE_ID = f"MOB_{MODEL_ID}_INT8_Full_Validation"
EXPERIMENT_ID = f"MOB_{MODEL_ID}_INT8_NonInferiority"
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
OUT = Path(os.environ.get("KLCN_OUTPUT_ROOT", "/kaggle/working")) / EXPERIMENT_ID
OUT.mkdir(parents=True, exist_ok=True)

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()
def read(path): return json.loads(Path(path).read_text(encoding="utf-8"))
def one(name):
    found=sorted(INPUT_ROOT.rglob(name)); assert found, (name, found)
    hashes={sha256(p) for p in found}; assert len(hashes)==1, (name, found)
    return found[0]
def write(path,obj): Path(path).write_text(json.dumps(obj,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")

RESULT_PATH=one(SOURCE_ID+"_RESULT.json")
CSV_PATH=one(SOURCE_ID+"_CAPTIONS.csv")
RESULT=read(RESULT_PATH)
assert RESULT["model_id"]==MODEL_ID
assert RESULT["status"]=="passed_with_caption_variation_review_required"
assert RESULT["test_set_used"] is False
assert all(v["passed"] for v in RESULT["gates"]["mixed_int8_quality"].values())
assert RESULT["gates"]["fp32_reproduction"]["passed"]
assert RESULT["gates"]["size"]["passed"]

if importlib.util.find_spec("pycocoevalcap") is None:
    wheels=sorted(INPUT_ROOT.rglob("pycocoevalcap*.whl")); assert wheels
    subprocess.check_call([sys.executable,"-m","pip","install","-q",str(wheels[0])])
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.cider.cider import Cider

df=pd.read_csv(CSV_PATH)
assert len(df)==1014 and df.image_id.nunique()==1014
refs={int(r.image_id):[{"caption":x} for x in str(r.references).split(" || ")] for r in df.itertuples()}
def cider_scores(column):
    hyps={int(r.image_id):[{"caption":getattr(r,column)}] for r in df.itertuples()}
    tok=PTBTokenizer(); rr=tok.tokenize(refs); hh=tok.tokenize(hyps)
    mean,scores=Cider().compute_score(rr,hh)
    return float(mean),np.asarray(scores,dtype=np.float64)
fp32_mean,fp32_scores=cider_scores("fp32_caption")
int8_mean,int8_scores=cider_scores("mixed_int8_caption")
delta=int8_scores-fp32_scores
rng=np.random.default_rng(42); draws=20000
bootstrap=np.empty(draws,dtype=np.float64)
for i in range(draws): bootstrap[i]=delta[rng.integers(0,len(delta),len(delta))].mean()
lower,upper=map(float,np.quantile(bootstrap,[.025,.975])); margin=-.01
passed=bool(lower>=margin)
payload={"schema_version":"mobile_int8_paired_noninferiority_v1",
 "created_at_utc":datetime.now(timezone.utc).isoformat(),"status":"passed" if passed else "failed",
 "experiment_id":EXPERIMENT_ID,"model_id":MODEL_ID,"teacher":TEACHER,"student":STUDENT,"method":METHOD,"seed":SEED,
 "split":"Flickr30K Karpathy Validation","images":len(df),"bootstrap_draws":draws,"bootstrap_seed":42,
 "metric":"CIDEr-D","fp32":fp32_mean,"mixed_int8":int8_mean,"paired_delta":float(delta.mean()),
 "confidence_interval_95":[lower,upper],"noninferiority_margin":margin,"passed":passed,
 "source":{"result_sha256":sha256(RESULT_PATH),"captions_sha256":sha256(CSV_PATH)},
 "validation_data_read":True,"test_set_used":False,
 "next_phase_authorized":f"MOB_{MODEL_ID}_Android_Lock" if passed else "STOP_REVIEW_NONINFERIORITY_FAILURE"}
result_path=OUT/f"{EXPERIMENT_ID}_RESULT.json"; write(result_path,payload)
write(OUT/"MANIFEST.json",{"schema_version":"mobile_noninferiority_manifest_v1","status":"complete",
 "artifacts":[{"relative_path":result_path.name,"bytes":result_path.stat().st_size,"sha256":sha256(result_path)}],"test_set_used":False})
print("Status                :",payload["status"].upper())
print("FP32 / INT8 CIDEr-D   :",fp32_mean,"/",int8_mean)
print("Paired delta          :",float(delta.mean()))
print("95% CI                :",[lower,upper])
print("Margin                :",margin)
print("Next                  :",payload["next_phase_authorized"])
assert passed,payload


PTBTokenizer tokenized 74030 tokens at 242718.12 tokens per second.
PTBTokenizer tokenized 10511 tokens at 89718.66 tokens per second.
PTBTokenizer tokenized 74030 tokens at 285635.56 tokens per second.
PTBTokenizer tokenized 10534 tokens at 78401.74 tokens per second.


Status                : PASSED
FP32 / INT8 CIDEr-D   : 0.3867191276728564 / 0.38485228601906674
Paired delta          : -0.0018668416537896211
95% CI                : [-0.006680956265703964, 0.0023059460066471712]
Margin                : -0.01
Next                  : MOB_F30K_T1S1_Seq_Android_Lock
